# Description

Projects the LINCS L1000 consensus drug signatures into the ARCHS4 CLAMP latent space.

Mirrors `phenoplier/nbs/30_drug_disease_associations/100-lincs/001-lincs_consensi-projection.ipynb` exactly, replacing `MultiplierProjection().transform()` with `CLAMP::projectCLAMP()`.

The input LINCS data (`lincs-data.pkl`) has already been processed (Entrez → Ensembl ID mapping, filtered to 7120 PhenomeXcan genes).

**Outputs** (`output/drug_disease_analyses/lincs/`):
- `lincs-data.pkl`: genes × drugs (Ensembl IDs, 7120 genes)
- `lincs-projection.pkl`: LVs × drugs (CLAMP projection)

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here

# Settings

In [3]:
# Input: processed LINCS data (Ensembl IDs, 7120 PhenomeXcan genes)
DATA_DIR = here('data/archs4/drug_diseases_associations')
LINCS_INPUT_FILE = DATA_DIR / 'lincs-data.pkl'
display(LINCS_INPUT_FILE)
assert LINCS_INPUT_FILE.exists()

CLAMP_MODEL_FILE = here('output/archs4/archs4_CLAMP_C2CP.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations/lincs-data.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/archs4_CLAMP_C2CP.rds')

In [4]:
OUTPUT_DIR = here('output/drug_disease_analyses') / 'lincs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs')

# Load LINCS data

In [5]:
lincs_data = pd.read_pickle(LINCS_INPUT_FILE)
display(lincs_data.shape)
display(lincs_data.head())
assert lincs_data.index.is_unique
assert lincs_data.columns.is_unique
assert not lincs_data.isna().any().any()

(7120, 1170)

perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
ENSG00000196839,-1.001,-1.835,1.391,1.132,0.257,1.932,0.508,1.408,0.777,0.032,...,-1.692,-0.516,-1.435,-0.317,-0.012,0.641,-0.230,-0.518,-0.177,2.146
ENSG00000170558,1.146,-1.863,0.011,-1.020,1.143,-0.115,1.327,0.310,-1.853,0.872,...,0.354,0.498,0.268,-1.084,-0.142,-0.077,0.633,-1.807,0.032,0.135
ENSG00000117020,-0.693,1.694,-0.804,-0.164,1.145,-1.465,1.221,-0.747,0.829,-0.961,...,-1.196,-0.230,-1.049,-0.347,0.586,0.865,-0.021,2.180,-0.956,0.105
ENSG00000133997,-0.037,0.383,0.269,-0.997,0.185,-0.536,0.424,-0.119,-1.313,0.579,...,-0.343,0.116,-0.245,-0.127,-1.367,0.149,0.117,2.084,1.178,0.772
ENSG00000101473,0.162,-0.899,0.105,-0.090,-1.291,1.404,0.185,0.157,-0.327,-0.026,...,-0.136,-1.115,-0.280,0.200,0.638,-0.197,-0.360,-2.302,-0.117,-0.167


In [6]:
# Verify all index entries are Ensembl IDs (15 chars)
_tmp = pd.Series(lincs_data.index.map(len)).value_counts()
display(_tmp)
assert _tmp.shape[0] == 1

15    7120
Name: count, dtype: int64

# Save raw LINCS data to output directory

In [7]:
output_raw_file = OUTPUT_DIR / 'lincs-data.pkl'
display(output_raw_file)
lincs_data.to_pickle(output_raw_file)
print('Saved.')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/lincs-data.pkl')

Saved.


# Load CLAMP model and prepare gene mapping

In [8]:
CLAMP = importr('CLAMP')
readRDS = ro.r['readRDS']
clamp = readRDS(str(CLAMP_MODEL_FILE))
print('CLAMP model loaded')

CLAMP model loaded


In [9]:
gene_symbols = list(ro.r['rownames'](clamp.rx2('Z')))
lv_names = list(ro.r['colnames'](clamp.rx2('Z')))
print(f'CLAMP genes: {len(gene_symbols)}, LVs: {len(lv_names)}')

CLAMP genes: 18423, LVs: 2366


In [10]:
# Map CLAMP gene symbols (HGNC) → Ensembl IDs
clusterProfiler = importr('clusterProfiler')

bitr_result = clusterProfiler.bitr(
    ro.StrVector(gene_symbols),
    fromType='SYMBOL',
    toType='ENSEMBL',
    OrgDb='org.Hs.eg.db',
)

with localconverter(ro.default_converter + pandas2ri.converter):
    mapping_df = ro.conversion.rpy2py(bitr_result)

print(f'Raw mapping shape: {mapping_df.shape}')
display(mapping_df.head())

R callback write-console: 
  


R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (19313, 2)


,SYMBOL,ENSEMBL
1,A1BG,ENSG00000121410
2,A1BG-AS1,ENSG00000268895
3,A2M,ENSG00000175899
4,A2M-AS1,ENSG00000245105
5,A2ML1,ENSG00000166535


In [11]:
# Keep only 1:1 unambiguous symbol ↔ Ensembl mappings
dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')
print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')

mapped_symbols = mapping_1to1.index.tolist()
mapped_ensembl = mapping_1to1['ENSEMBL'].tolist()

1:1 mappings: 16038 / 18423 CLAMP genes


In [12]:
# Build CLAMP sub-object with Z restricted to 1:1-mapped genes.
# as.matrix() ensures dense numeric matrix for projectCLAMP's %*% operator.
subset_Z = ro.r('function(clamp, genes) { clamp$Z <- as.matrix(clamp$Z[genes, ]); clamp }')
clamp_sub = subset_Z(clamp, ro.StrVector(mapped_symbols))
print(f'Subsetted CLAMP Z: {len(mapped_symbols)} genes x {len(lv_names)} LVs')

R callback write-console: In addition:   


R callback write-console: Warning message:
  


R callback write-console: In (function (geneID, fromType, toType, OrgDb, drop = TRUE)  :  


R callback write-console: 
   


R callback write-console:  6.63% of input gene IDs are fail to map...
  


Subsetted CLAMP Z: 16038 genes x 2366 LVs


# Project LINCS into CLAMP

In [13]:
# Align LINCS to CLAMP gene order (mapped Ensembl IDs), fill missing genes with 0
aligned = lincs_data.reindex(mapped_ensembl).fillna(0.0).values  # (n_genes, n_drugs)

r_mat = ro.r['matrix'](
    ro.FloatVector(aligned.flatten('F')),
    nrow=aligned.shape[0],
    ncol=aligned.shape[1],
)

proj_r = CLAMP.projectCLAMP(clamp_sub, newdata=r_mat)

with localconverter(ro.default_converter + pandas2ri.converter):
    proj_values = ro.conversion.rpy2py(proj_r)

lincs_projection = pd.DataFrame(proj_values, index=lv_names, columns=lincs_data.columns)
print(f'LINCS projection shape: {lincs_projection.shape}')
display(lincs_projection.head())

LINCS projection shape: (2366, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,-0.003664,0.021802,0.050683,-0.023411,0.054818,0.022504,-0.016074,-0.007949,-0.012020,-0.022674,...,-0.051500,-0.039418,-0.021987,0.012250,0.009535,0.012377,0.013959,-0.209899,0.026002,0.009327
LV2,0.011797,0.058622,0.001288,-0.023493,0.009715,0.001838,-0.025821,-0.026784,-0.010831,0.000237,...,-0.003300,-0.000286,0.004965,0.011235,0.002291,-0.003366,-0.002029,-0.067440,0.005340,0.011255
LV3,-0.000841,-0.055107,-0.006764,0.027218,-0.012003,-0.010472,-0.021370,-0.007238,-0.005116,0.005954,...,-0.003723,0.009973,-0.007469,0.009228,-0.010771,-0.017754,0.012783,0.066089,0.008973,-0.011303
LV4,0.031992,-0.511423,-0.071332,-0.059371,-0.035587,-0.033645,-0.076674,0.042806,-0.099315,0.031507,...,0.051060,0.022656,0.054687,0.014682,-0.078138,-0.014696,0.015666,-0.238956,-0.018649,-0.047013
LV5,-0.032510,0.123403,0.005948,0.031886,-0.045581,-0.006246,0.092339,0.034634,0.111000,-0.025291,...,-0.020998,-0.019916,-0.030282,0.009627,0.031024,-0.017982,-0.019684,0.164316,0.012215,-0.027974


In [14]:
assert not lincs_projection.isna().any().any()

# Save

In [15]:
output_proj_file = OUTPUT_DIR / 'lincs-projection.pkl'
display(output_proj_file)
lincs_projection.to_pickle(output_proj_file)
print('Saved.')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/lincs-projection.pkl')

Saved.
